# Competição · 5 · métricas de negócio (compra de testes)

**Pergunta:** quantos testes comprar para o período, de forma que **não falte**, mas também **não sobre demais**?

**Não precisa rodar nenhum modelo de novo.** Este notebook lê os quantis que os notebooks 1 a 4 já gravaram em `resultados_competicao/`, reconstrói trajetórias e calcula as métricas. **Ambiente:** CPU padrão; leva alguns minutos.

**Como a compra é decidida (problema do jornaleiro):** compra-se o quantil **τ\* = custo de faltar ÷ (custo de faltar + custo de sobrar)** da previsão do **total do período**. Exemplo: faltar custa 3× mais que sobrar → τ\* = 0,75 → compra-se o percentil 75 do total (≈ 25% de chance de faltar).

**O total do período não é a soma dos quantis semanais**, porque somar os P75 de cada semana superestima o total. Por isso o notebook transforma os quantis em trajetórias (o mesmo método da soma das regiões: amostras + reordenação de Schaake com a dependência histórica entre semanas e unidades) e soma as trajetórias.

| Métrica | O que mede | Bom quando |
|---|---|---|
| `custo_vs_climatologia` | custo real da compra (falta × custo de faltar + sobra × custo de sobrar) ÷ o mesmo custo usando a climatologia | **< 1** · **é a métrica para escolher o modelo** |
| `falta_freq_pct` | % das unidades (UF ou região) × temporadas em que **faltou** teste | perto de (1 − τ\*) × 100 |
| `falta_pct` / `sobra_pct` | testes que faltaram / sobraram, em % do total real | baixos |
| `WAPE_total_pct` | erro da mediana do total, em % do total real | baixo |
| `vies_total_pct` | a mediana do total está acima (+) ou abaixo (−) do real | perto de 0 |
| `MAE_semanal` | erro médio da mediana, em casos por unidade e semana (dominado pelas UFs grandes) | baixo |
| `WAPE_semanal_pct` | soma dos erros semanais ÷ soma dos casos (o "erro percentual" estável) | baixo |
| `sMAPE_semanal_pct` | erro percentual simétrico (0–200%); instável com zeros, ignora os pares em que real e previsto são 0 | baixo, mas leia com cuidado |
| `WIS_semanal`, `IC90_semanal_pct` | métrica do desafio e cobertura do intervalo de 90% | WIS baixo; IC90 ≈ 90 |

⚠️ Uma temporada só (por exemplo, só a S1) tem pouca amostra: 26 UFs. Confira depois com as 4 temporadas.

In [ ]:
NOTEBOOK = 'negocio'
PACOTES = ''

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/comp', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'dados', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM
for m in (V, L, CM):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = L.Contexto(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'))
TODAS = list(range(2016, 2026))          # 2016–2021: calibração dos pesos do ensemble · 2022–2025: S1–S4
RECENTES = list(range(2020, 2026))       # para os modelos mais caros
OFICIAIS = V.TEMPORADAS_OFICIAIS
M = 500                                  # trajetórias por região
REFAZER = False

## Configuração

In [ ]:
TEMPORADAS_AVAL = [2022]          # S1 = 2022 (SE41/2022 → SE40/2023). Para as 4: [2022, 2023, 2024, 2025]
PERIODOS = {                      # índices das semanas da temporada (0 = SE41); None = até o fim
    'temporada (SE41→SE40)': (0, None),
    '1º semestre (SE41→SE14)': (0, 26),
    '2º semestre (SE15→SE40)': (26, None),
}
CUSTO_FALTA, CUSTO_SOBRA = 3.0, 1.0      # faltar 1 teste custa 3× mais que sobrar 1 → τ* = 0,75
TESTES_POR_CASO = 1.0                    # converte casos notificados em testes (ajuste se souber a taxa real)
M_NEG = 500                              # trajetórias por unidade
INCLUIR_IMDC = True                      # equipes do 3º IMDC e BB oficial (só por UF; só nas temporadas S1–S4)
INCLUIR_ENSEMBLES = True                 # ensembles do Comp4 e do CompUF4 (só por UF)
PASTAS_EXTRA = ([p for p in sorted(glob.glob(os.path.join(PASTA_DRIVE, 'resultados_uf*'))) if os.path.isdir(p)]  # CompUF1–4
                + [os.path.join(PASTA_DRIVE, 'resultados_externos')])  # modelos de fora (L.importar_externo); pode não existir
# Escolha honesta do modelo para a empresa: escolhe em AFINACAO, confirma em TESTE (veja a seção 5).
# Para usá-la, ponha em TEMPORADAS_AVAL as temporadas das duas listas, ex.: list(range(2018, 2026)).
AFINACAO, TESTE = [2018, 2019, 2020, 2021], [2022, 2023, 2024, 2025]

TAU = CUSTO_FALTA / (CUSTO_FALTA + CUSTO_SOBRA)
TAU_GRADE = sorted(set(np.round(np.arange(0.05, 0.96, 0.05), 3)) | {round(TAU, 3)})
PASTA_NEG = os.path.join(PASTA_RES, 'negocio'); os.makedirs(PASTA_NEG, exist_ok=True)
SIGLA = {11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO', 21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN',
         25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA', 31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR', 42: 'SC',
         43: 'RS', 50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'}
print(f'τ* = {TAU:.3f} → compra-se o percentil {100 * TAU:.0f} do total previsto')

In [ ]:
# @title Funções: trajetórias a partir dos quantis e métricas por unidade
def template(D, s, M, a=0.85, seed=0):
    # igual a L.template_dependencia, mas para qualquer nível (região ou UF)
    rng = np.random.default_rng(seed + s)
    c = D.corte(s)
    W = len(V.datas_temporada(s))
    temps = [j for j in range(2010, s) if D.temporada_completa(j, c) and D.idx_temporada(j)[0] > 0]
    F = []
    for j in temps:
        x = D.LINC[D.idx_temporada(j)]
        F.append(np.vstack([x, np.repeat(x[-1:], max(0, W - len(x)), 0)])[:W])
    F = np.stack(F)
    A = (F - F.mean(0)) / (F.std(0) + 1e-6)
    js = rng.integers(0, len(temps), M)
    return (a * A[js] + np.sqrt(1 - a ** 2) * rng.standard_normal((M, W, D.N))).astype(np.float32)


def trajetorias(q, D, s, M):
    am = L.amostrar_quantis(q, M)                        # (M, W, N), marginais exatas de cada semana
    return L.schaake(am, template(D, s, M)[:, :q.shape[0]])


def metricas_unidades(q, Y, D, ids, s):
    # q (W, N, 9) e Y (W, N) da temporada inteira → DataFrame com uma linha por unidade e período
    traj = trajetorias(q, D, s, M_NEG)
    saida = []
    for per, (a, b) in PERIODOS.items():
        qq, yy, tt = q[a:b], Y[a:b], traj[:, a:b]
        obs = ~np.isnan(yy[:, 0])
        if obs.sum() == 0:
            continue
        med, yo = qq[obs, :, 4], yy[obs]
        err = med - yo
        den = np.abs(yo) + np.abs(med)
        tem = den > 0
        smape = np.where(tem, 2 * np.abs(err) / np.where(tem, den, 1), 0)
        cob = (yo >= qq[obs, :, 1]) & (yo <= qq[obs, :, 7])
        T = tt[:, obs].sum(1)                            # (M, N): total do período em cada trajetória
        d = pd.DataFrame({'unidade': ids, 'season': s, 'periodo': per, 'completo': bool(obs.all()),
                          'n_sem': int(obs.sum()), 'soma_y': yo.sum(0), 'soma_abs': np.abs(err).sum(0),
                          'soma_err': err.sum(0), 'soma_smape': smape.sum(0), 'n_smape': tem.sum(0),
                          'soma_wis': V.wis(qq[obs], yo).sum(0), 'n_cob90': cob.sum(0),
                          'y_total': yo.sum(0), 'mediana_total': np.median(T, 0)})
        qT = np.quantile(T, TAU_GRADE, axis=0)
        for k, t in enumerate(TAU_GRADE):
            d[f'q{t:.3f}'] = qT[k]
        saida.append(d)
    return pd.concat(saida, ignore_index=True)

## 1. Calcular (alguns minutos)

In [ ]:
# Leitura padronizada: aceita qualquer modelo que tenha gravado os 9 quantis (nossos, ensembles, externos).
# Duplicatas → fica a fonte listada por último; previsões incompletas ficam fora (comparação justa). Ver o relatório.
fontes_uf = [('competição', PASTA_RES)]
if INCLUIR_IMDC and ctx.ref is not None:
    fontes_uf.append(('3º IMDC', ctx.ref))
if INCLUIR_ENSEMBLES:
    fontes_uf += [('ensembles', os.path.join(p, 'ensembles_uf.parquet')) for p in [PASTA_RES] + PASTAS_EXTRA]
for p in PASTAS_EXTRA:
    fontes_uf += [(os.path.basename(p), p), ('final', os.path.join(p, 'final_uf.parquet'))]
PU = L.carregar_previsoes(fontes_uf, ctx.Du.ufs, 'uf')
PR = L.carregar_previsoes([('competição', PASTA_RES)], ctx.Dr.unidades, 'regiao')
PU, PR = PU[PU.season.isin(TEMPORADAS_AVAL)], PR[PR.season.isin(TEMPORADAS_AVAL)]

NIVEIS = {'UF': (PU, ctx.Du, ctx.Du.ufs, ctx.ver_uf), 'região': (PR, ctx.Dr, ctx.Dr.unidades, ctx.ver_reg)}
partes, pulados = [], []
for nivel, (P, D, ids, ver) in NIVEIS.items():
    for (m, s), g in P.groupby(['modelo', 'season']):
        datas = V.datas_temporada(s)
        q = V.de_df(g, datas, ids)
        if np.isnan(q).any():                         # semanas ou unidades faltando: fica de fora (comparação justa)
            pulados.append((nivel, m, s, int(np.isnan(q[..., 4]).sum())))
            continue
        d = metricas_unidades(V.arruma_quantis(q), ver.y(datas), D, ids, s)
        d.insert(0, 'modelo', m); d.insert(0, 'nivel', nivel)
        partes.append(d)
    print(f'{nivel}: {P.modelo.nunique()} modelos', flush=True)
U = pd.concat(partes, ignore_index=True)
U.to_parquet(os.path.join(PASTA_NEG, 'metricas_por_unidade.parquet'), index=False)
if pulados:
    print('Ficaram de fora (previsão incompleta):'); display(pd.DataFrame(pulados, columns=['nivel', 'modelo', 'season', 'células vazias']))
print('Linhas:', len(U), '· temporadas:', sorted(U.season.unique()))

In [ ]:
# @title Resumo por modelo (escolha do modelo = menor custo_vs_climatologia)
CHAVE = ['nivel', 'periodo', 'season', 'unidade']


def resumo(U, tau):
    c = f'q{tau:.3f}'
    U = U.copy()
    U['_falta'] = np.maximum(U.y_total - U[c], 0)
    U['_sobra'] = np.maximum(U[c] - U.y_total, 0)
    U['_custo'] = (CUSTO_FALTA * U._falta + CUSTO_SOBRA * U._sobra) * TESTES_POR_CASO
    U['_faltou'] = (U.y_total > U[c]).astype(float)
    U['_ae_tot'] = (U.mediana_total - U.y_total).abs()
    ref = U[U.modelo == 'climatologia'][CHAVE + ['_custo']].rename(columns={'_custo': '_custo_ref'})
    U = U.merge(ref, on=CHAVE, how='left')
    lin = []
    for (nv, per, m), g in U.groupby(['nivel', 'periodo', 'modelo']):
        gc = g[g.completo]
        gr = gc[gc._custo_ref.notna()]
        ytot = gc.y_total.sum()
        lin.append(dict(
            nivel=nv, periodo=per, modelo=m, temporadas=g.season.nunique(), unidades=g.unidade.nunique(),
            custo_vs_climatologia=gr._custo.sum() / gr._custo_ref.sum() if len(gr) else np.nan,
            falta_freq_pct=100 * gc._faltou.mean(), falta_pct=100 * gc._falta.sum() / ytot,
            sobra_pct=100 * gc._sobra.sum() / ytot, WAPE_total_pct=100 * gc._ae_tot.sum() / ytot,
            vies_total_pct=100 * (gc.mediana_total.sum() - ytot) / ytot,
            MAE_semanal=g.soma_abs.sum() / g.n_sem.sum(), WAPE_semanal_pct=100 * g.soma_abs.sum() / g.soma_y.sum(),
            sMAPE_semanal_pct=100 * g.soma_smape.sum() / max(g.n_smape.sum(), 1),
            vies_semanal_pct=100 * g.soma_err.sum() / g.soma_y.sum(),
            WIS_semanal=g.soma_wis.sum() / g.n_sem.sum(), IC90_semanal_pct=100 * g.n_cob90.sum() / g.n_sem.sum(),
            custo_total=gc._custo.sum()))
    return pd.DataFrame(lin).sort_values(['nivel', 'periodo', 'custo_vs_climatologia'])


R = resumo(U, round(TAU, 3))
R.to_csv(os.path.join(PASTA_NEG, f'resumo_tau{TAU:.2f}.csv'), index=False)
print(f'τ* = {TAU:.2f} · falta_freq ideal ≈ {100 * (1 - TAU):.0f}% · temporadas: {TEMPORADAS_AVAL}')
for (nv, per), g in R.groupby(['nivel', 'periodo'], sort=False):
    print(f'\n=== {nv} · {per} ===')
    display(g.drop(columns=['nivel', 'periodo', 'custo_total']).round(3).head(20))

## 2. Os mesmos números para outros τ

Se você não sabe o custo de faltar em relação ao de sobrar, compare os modelos em vários τ. Um modelo que vence em todos é a escolha robusta.

In [ ]:
PER0 = list(PERIODOS)[0]
tabs = []
for t in (0.5, 0.75, 0.9):
    r = resumo(U, t)
    r = r[r.periodo == PER0][['nivel', 'modelo', 'custo_vs_climatologia', 'falta_freq_pct', 'sobra_pct']]
    tabs.append(r.set_index(['nivel', 'modelo']).add_suffix(f' τ={t}'))
T3 = pd.concat(tabs, axis=1).reset_index().sort_values(['nivel', 'custo_vs_climatologia τ=0.75'])
T3.to_csv(os.path.join(PASTA_NEG, 'comparacao_taus.csv'), index=False)
for nv, g in T3.groupby('nivel'):
    print(f'\n=== {nv} · {PER0} ==='); display(g.drop(columns='nivel').round(2).head(20))

## 3. Gráficos

- **Falta × sobra (τ\*):** cada ponto é um modelo. O canto inferior esquerdo é o ideal. A linha tracejada marca a frequência de falta esperada (1 − τ\*).
- **Calibração do total:** para cada τ, a % de vezes que faltou. Um modelo bem calibrado fica sobre a diagonal (falta = 1 − τ). Abaixo dela, superestima e sobra teste; acima, subestima e falta teste.

In [ ]:
import matplotlib.pyplot as plt
for nv in R.nivel.unique():
    g = R[(R.nivel == nv) & (R.periodo == PER0)].dropna(subset=['custo_vs_climatologia'])
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
    cor = np.where(g.modelo.str.startswith('IMDC:') | (g.modelo == 'BB'), 'lightgray',
                   np.where(g.modelo.str.startswith('ENS'), 'tab:purple', 'tab:red'))
    ax[0].scatter(g.sobra_pct, g.falta_freq_pct, c=cor, s=40)
    for _, r in g.nsmallest(12, 'custo_vs_climatologia').iterrows():
        ax[0].annotate(r.modelo.replace('IMDC:', ''), (r.sobra_pct, r.falta_freq_pct), fontsize=7)
    ax[0].axhline(100 * (1 - TAU), ls='--', c='k', lw=.8)
    ax[0].set(xlabel='testes que sobraram (% do total real)', ylabel='% de unidades em que faltou',
              title=f'{nv} · {PER0} · τ*={TAU:.2f} (vermelho = nossos; cinza = 3º IMDC; roxo = ensembles)')
    top = g.nsmallest(6, 'custo_vs_climatologia').modelo.tolist()
    if 'climatologia' not in top:
        top.append('climatologia')
    taus = [t for t in TAU_GRADE if 0.4 <= t <= 0.96]
    for m in top:
        u = U[(U.nivel == nv) & (U.periodo == PER0) & (U.modelo == m) & U.completo]
        ax[1].plot(taus, [100 * (u.y_total > u[f'q{t:.3f}']).mean() for t in taus], marker='o', ms=3, label=m)
    ax[1].plot(taus, [100 * (1 - t) for t in taus], 'k--', lw=1, label='ideal')
    ax[1].set(xlabel='τ (quantil comprado)', ylabel='% de vezes em que faltou', title='calibração do total do período')
    ax[1].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(PASTA_NEG, f'graficos_{nv}.png'), dpi=120); plt.show()

## 4. Compra sugerida × consumo real, por UF (melhor modelo)

In [ ]:
MODELO_COMPRA = None        # None = o de menor custo_vs_climatologia por UF; ou, por exemplo, 'ARIMA_Fourier'
g = R[(R.nivel == 'UF') & (R.periodo == PER0)].dropna(subset=['custo_vs_climatologia'])
m = MODELO_COMPRA or g.iloc[0].modelo
c = f'q{round(TAU, 3):.3f}'
for s in TEMPORADAS_AVAL:
    u = U[(U.nivel == 'UF') & (U.periodo == PER0) & (U.modelo == m) & (U.season == s)].copy()
    if not len(u):
        continue
    t = pd.DataFrame({'UF': u.unidade.map(SIGLA),
                      'real': (u.y_total * TESTES_POR_CASO).round(),
                      'compra (τ*)': (u[c] * TESTES_POR_CASO).round(),
                      'mediana prevista': (u.mediana_total * TESTES_POR_CASO).round()})
    t['diferença %'] = (100 * (t['compra (τ*)'] - t.real) / t.real.clip(lower=1)).round(0)
    t['faltou?'] = np.where(t.real > t['compra (τ*)'], 'FALTOU', '')
    t = t.sort_values('real', ascending=False)
    t.to_csv(os.path.join(PASTA_NEG, f'compra_por_uf_{m}_{s}.csv'.replace(':', '_').replace('/', '_')), index=False)
    print(f'\n{m} · temporada {s} · {PER0} · τ*={TAU:.2f} · faltou em {(t["faltou?"] != "").sum()} de {len(t)} UFs')
    print(f'Total Brasil: real {t.real.sum():,.0f} · compra {t["compra (τ*)"].sum():,.0f} '
          f'({100 * (t["compra (τ*)"].sum() / t.real.sum() - 1):+.0f}%)')
    display(t)
print('Arquivos em', PASTA_NEG, ':', sorted(os.listdir(PASTA_NEG)))

## 5. Escolha honesta para a empresa: escolhe na afinação, confirma no teste

Escolher o modelo pelas mesmas temporadas em que se mede o resultado dá números otimistas. Aqui o modelo é escolhido pelo custo em `AFINACAO` e só depois o custo em `TESTE` é mostrado. Precisa de `TEMPORADAS_AVAL` com as duas listas; só entram os modelos que rodaram em todas as temporadas de cada lista.

In [ ]:
tem = set(U.season.unique())
if not (set(AFINACAO) <= tem and set(TESTE) <= tem):
    print('Ponha em TEMPORADAS_AVAL as temporadas de AFINACAO e TESTE, ex.: list(range(2018, 2026)), e rode de novo.')
else:
    ra = resumo(U[U.season.isin(AFINACAO)], round(TAU, 3))
    rt = resumo(U[U.season.isin(TESTE)], round(TAU, 3))
    ra, rt = ra[ra.temporadas == len(AFINACAO)], rt[rt.temporadas == len(TESTE)]
    col = ['custo_vs_climatologia', 'falta_freq_pct', 'sobra_pct', 'WAPE_total_pct']
    chave = ['nivel', 'periodo', 'modelo']
    H = ra[chave + col].merge(rt[chave + col], on=chave, suffixes=(' · afinação', ' · teste'))
    H.to_csv(os.path.join(PASTA_NEG, f'escolha_afinacao_teste_tau{TAU:.2f}.csv'), index=False)
    for (nv, per), g in H.groupby(['nivel', 'periodo'], sort=False):
        g = g.sort_values('custo_vs_climatologia · afinação')
        print(f'\n=== {nv} · {per} · escolhido na afinação: {g.iloc[0].modelo} ===')
        display(g.drop(columns=['nivel', 'periodo']).round(3).head(15))

## 6. As métricas entre si: elas escolhem o mesmo modelo?

Cada métrica ordena os modelos. Se a correlação entre as ordens for alta, tanto faz qual otimizar; se for baixa, a escolha da métrica muda o modelo escolhido, e para a compra deve valer o **custo** (o da empresa). `WIS_semanal` é a régua da competição; `custo_vs_climatologia` é a da compra.

In [ ]:
METRICAS = {'custo_vs_climatologia': True, 'WIS_semanal': True, 'MAE_semanal': True, 'WAPE_semanal_pct': True,
            'sMAPE_semanal_pct': True, 'WAPE_total_pct': True, 'falta_pct': True, 'sobra_pct': True,
            'abs_vies_total_pct': True, 'erro_IC90': True}           # True = menor é melhor
for (nv, per), g in R.groupby(['nivel', 'periodo'], sort=False):
    g = g.dropna(subset=['custo_vs_climatologia']).copy()
    if len(g) < 4:
        continue
    g['abs_vies_total_pct'] = g.vies_total_pct.abs()
    g['erro_IC90'] = (g.IC90_semanal_pct - 90).abs()                    # distância da cobertura ideal
    rk = pd.DataFrame({m: g[m].rank(ascending=asc) for m, asc in METRICAS.items()}).set_index(g.modelo)
    print(f'\n=== {nv} · {per} · {len(g)} modelos ===')
    print('Correlação de Spearman entre as ordens (1 = escolhem igual):')
    display(rk.corr(method='spearman').round(2))
    print('Vencedor por métrica:')
    display(pd.DataFrame({m: [rk[m].idxmin()] for m in METRICAS}, index=['1º lugar']).T)
    rk.to_csv(os.path.join(PASTA_NEG, f'posicoes_por_metrica_{nv}_{per[:9].strip()}.csv'.replace(' ', '_')))